# EDA - Controle do Tabagismo (OMS)

Projeto de Ciência de Dados - Etapa 2

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
BASE_URL = "https://ghoapi.azureedge.net/api"

def buscar_indicadores(termo="Tobacco"):
    url = f"{BASE_URL}/Indicator?$filter=contains(IndicatorName,'{termo}')"
    return pd.DataFrame(requests.get(url).json()["value"])

def baixar_indicador(codigo):
    url = f"{BASE_URL}/{codigo}"
    return pd.DataFrame(requests.get(url).json()["value"])

In [ ]:
indicadores = buscar_indicadores("Tobacco")
indicadores[["IndicatorCode","IndicatorName"]].head(10)

In [ ]:
codigo = "gho_tobacco_control_monitor_current_tobaccouse_tobaccosmoking_cigarrettesmoking_agestd_tobagestdcurr"
df = baixar_indicador(codigo)
df.head()

In [ ]:
df = df.rename(columns={
    "SpatialDim": "pais",
    "TimeDim": "ano",
    "NumericValue": "valor"
})

df["ano"] = pd.to_numeric(df["ano"], errors="coerce")
df["valor"] = pd.to_numeric(df["valor"], errors="coerce")

df = df.dropna(subset=["pais","ano","valor"])
df.head()

In [ ]:
ano_mais_recente = int(df["ano"].max())
df_recente = df[df["ano"] == ano_mais_recente]

print("Ano:", ano_mais_recente)

In [ ]:
df_recente.describe()

In [ ]:
plt.hist(df_recente["valor"], bins=20)
plt.title("Distribuição da prevalência")
plt.show()

In [ ]:
top10 = df_recente.sort_values("valor", ascending=False).head(10)

plt.barh(top10["pais"], top10["valor"])
plt.gca().invert_yaxis()
plt.title("Top 10 países")
plt.show()

In [ ]:
df.to_csv("data/processed/dados_tratados.csv", index=False)
df_recente.to_csv("data/processed/dados_recente.csv", index=False)

In [ ]:
print("Próxima etapa: aplicar modelo de classificação (Árvore de Decisão)")